<a href="https://colab.research.google.com/github/adib422/FlyRank-Internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring** (Lane 2).

I'm choosing this lane because it directly extends the workflow I already ran in the starter notebooks: Notebook 1 showed that search volume doesn't guarantee visibility and that CTR drops sharply below the top positions, and Notebook 2 showed that a shallow, readable model can meaningfully beat a hand-written staleness rule once validated honestly with a client holdout. This lane lets me push that same question further, not just "does a page look stale," but "which pages, out of everything a client owns, should an editor actually look at first, this week, given they can only review a limited number." It also has the most direct connection to an action a real person can take (review, refresh, expand, prune, or monitor), which is what the framing skill asks me to name.


In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [4]:
# Load the starter dataset once here so section 3 can reuse it.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)


(30000, 44)


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The question: decision, action, cost of a wrong call

**Research question:** Of all the content a client owns, which pages should an editor review first for refresh, expansion, pruning, or monitoring, given they only have time to check a limited number each week?

**Decision this improves:** which pages land at the top of a content editor's weekly review queue.

**Who acts, and what do they do:** a content editor / SEO strategist with fixed review capacity (they can realistically only look closely at a few dozen pages a week). They open each flagged page, check the reason code(s) attached to it, and decide to refresh, expand, prune, or leave it monitored.

**Unit of analysis:** one content page (`content_id`), scored using its trailing 90-day window of signals.

**Cost of a wrong call:**
- **False positive** (flagged as worth reviewing, but it's actually fine): wastes an editor's limited hour on a page that didn't need attention -- a real but recoverable cost, since capacity used here just isn't used somewhere better.
- **False negative** (a genuinely declining, high-demand page never gets flagged): worse, because organic decline compounds -- lost rankings are harder to win back the longer they go unnoticed, and the page keeps leaking traffic silently every day it isn't reviewed.
Because a missed decline is more expensive than a wasted review, I'll care about **precision in the top of the ranking (precision@K)** more than raw accuracy -- the queue only has to be right near the top.

**Why data/ML helps instead of a hand-written rule:** a page's "worth reviewing" status depends on several signals moving together (staleness, visibility, position, CTR, engagement) in ways that shift by content type and traffic tier -- too tangled to hand-write confidently, but Notebook 2 already showed a simple, readable model can learn a better ranking than a single if-else rule once honestly validated.


In [7]:
# No computation needed here -- section 2 is framing. Left intentionally as a pass-through cell.
print("Decision: prioritize editor's weekly review queue. Actor: content editor. Cost asymmetry: missed decline > wasted review.")


Decision: prioritize editor's weekly review queue. Actor: content editor. Cost asymmetry: missed decline > wasted review.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [5]:
# Number 1: how much of the inventory is already declining -- too much to review by hand
base_rate = df["trend_direction"].str.lower().eq("down").mean()
print(f"1) Base rate of declining pages: {base_rate:.1%} ({int(base_rate*len(df))} of {len(df)} pages)")
print("   -- over half the inventory shows decline; no editor can manually triage that much, so ranking matters.")
print()

# Number 2: how many of those declining pages still have real search demand (i.e. worth fixing, not dead pages)
declining_with_demand = (df["trend_direction"].str.lower().eq("down")) & (df["impressions_90d"] >= 100)
print(f"2) declining_with_demand (declining AND impressions_90d >= 100): {declining_with_demand.sum()} pages ({declining_with_demand.mean():.1%})")
print("   -- these are pages actively losing ground while still being searched for: the highest-value review candidates.")
print()

# Number 3: baseline rule vs a learned ranking, verified from the starter pipeline's own output
print("3) From outputs/model_results.json (starter pipeline, client-holdout validated):")
print("   baseline rule      Precision@50 = 0.240  (~12 of top 50 correct)")
print("   random forest      Precision@50 = 0.740  (~37 of top 50 correct)")
print("   -- a learned ranking roughly triples how many of the top 50 recommendations are real, on this starter slice.")


1) Base rate of declining pages: 54.2% (16262 of 30000 pages)
   -- over half the inventory shows decline; no editor can manually triage that much, so ranking matters.

2) declining_with_demand (declining AND impressions_90d >= 100): 13152 pages (43.8%)
   -- these are pages actively losing ground while still being searched for: the highest-value review candidates.

3) From outputs/model_results.json (starter pipeline, client-holdout validated):
   baseline rule      Precision@50 = 0.240  (~12 of top 50 correct)
   random forest      Precision@50 = 0.740  (~37 of top 50 correct)
   -- a learned ranking roughly triples how many of the top 50 recommendations are real, on this starter slice.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful words: what I can and can't claim

**What this work CAN say:**
- Which pages, based on *observed* trailing signals, look most similar to pages that were later measured as declining -- an *observed, directional* pattern, not a certainty.
- That a learned ranking beat a hand-written rule at precision@K on this dataset, validated with a client holdout so it isn't just memorizing clients it already saw.
- That flagged pages are *decision-support*: a starting point for a human reviewer's limited time, not a verdict.

**What this work CANNOT say:**
- That refreshing a flagged page *causes* it to recover -- proving that needs a real experiment (e.g. before/after with a control group), not this dataset.
- Anything about *why* Google ranks a page the way it does -- I only have observed outcomes (impressions, clicks, position), never the algorithm itself.
- That a "declining" label is guaranteed real decline and not consolidation, seasonality, or noise -- I'll need to rule those out explicitly before trusting any single flagged page (per the lane guide's section on decline vs. look-alikes).
- Nothing about individual clients, URLs, or queries -- all IDs stay pseudonymous in anything I write up publicly.


In [6]:
# No computation needed -- section 4 is a claims/scope statement, not a numeric check.
print("Claims allowed: observed, directional, decision-support. Claims NOT allowed: causal, algorithmic, client-identifying.")


Claims allowed: observed, directional, decision-support. Claims NOT allowed: causal, algorithmic, client-identifying.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.